# Dummy HAB classifier

This notebook establishes a majority-class baseline using `thetao_mean` as the only feature. The classifier is fit on `hab_train.csv` and evaluated on the held-out `hab_validation.csv` using the `target` column, where 1 indicates a HAB event. The Copernicus physics file is unlabeled, so predictions on it are only a pipeline smoke test.

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

In [ ]:
train = pd.read_csv('../data/processed/hab_train.csv')
validation = pd.read_csv('../data/processed/hab_validation.csv')

X_train = train[['thetao_mean']]
y_train = train['target'].astype(int)
X_validation = validation[['thetao_mean']]
y_validation = validation['target'].astype(int)

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_validation)

print(f'Training rows: {len(X_train)}')
print(f'Validation rows: {len(X_validation)}')
print(f'Training class counts:\n{y_train.value_counts().sort_index()}')
print(f'Predicted class: {baseline.classes_[baseline.class_prior_.argmax()]}')
print(f'Validation accuracy: {accuracy_score(y_validation, y_pred):.3f}')
print(f'Validation balanced accuracy: {balanced_accuracy_score(y_validation, y_pred):.3f}')
print(classification_report(
    y_validation,
    y_pred,
    target_names=['Non-event', 'HAB event'],
    zero_division=0,
))

## Smoke test on unlabeled Copernicus features

These rows do not have `target`, so accuracy cannot be calculated. Since this is a dummy classifier, every row receives the same majority-class prediction.

In [ ]:
copernicus = pd.read_csv('../data/processed/copernicus_physics_features.csv')
copernicus['predicted_target'] = baseline.predict(copernicus[['thetao_mean']])
copernicus['predicted_hab'] = copernicus['predicted_target'].eq(1)

print('This is a pipeline check, not an accuracy evaluation.')
display(copernicus[['haedat_row_id', 'thetao_mean', 'predicted_hab']].head())